In [ ]:
pip install torch transformers librosa numpy pandas scikit-learn

In [3]:
import os
import librosa
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

# Load metadata
metadata_path = "./archive/UrbanSound8K.csv"
metadata = pd.read_csv(metadata_path)

# Create file paths and labels
file_paths = []
labels = []

for index, row in metadata.iterrows():
    file_path = os.path.join("./archive", f"fold{row['fold']}", row["slice_file_name"])
    file_paths.append(file_path)
    labels.append(row["classID"])

# Split into train and test sets
train_file_paths, test_file_paths, y_train, y_test = train_test_split(file_paths, labels, test_size=0.2, random_state=42)

# Function to extract features
def extract_features(file_path, n_mfcc=40, max_pad_len=174, n_fft=512):
    try:
        audio, sample_rate = librosa.load(file_path, sr=22050)
        mfccs = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=n_mfcc, n_fft=n_fft)

        # Pad or truncate to ensure fixed shape
        if mfccs.shape[1] < max_pad_len:
            pad_width = max_pad_len - mfccs.shape[1]
            mfccs = np.pad(mfccs, ((0, 0), (0, pad_width)), mode='constant')
        else:
            mfccs = mfccs[:, :max_pad_len]

        return mfccs.T  # Transpose for CNN input
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

# Apply feature extraction in batches to save memory
def batch_extract_features(file_paths, batch_size=1000):
    features = []
    for start in range(0, len(file_paths), batch_size):
        end = min(start + batch_size, len(file_paths))
        batch_file_paths = file_paths[start:end]
        batch_features = [extract_features(fp) for fp in batch_file_paths]
        features.extend([f for f in batch_features if f is not None])
    return np.array(features)

# Extract features for training and testing sets
X_train_features = batch_extract_features(train_file_paths)
X_test_features = batch_extract_features(test_file_paths)

# Apply PCA for dimensionality reduction
n_components = 40
pca = PCA(n_components=n_components)
X_train_pca = pca.fit_transform(X_train_features.reshape(X_train_features.shape[0], -1))
X_test_pca = pca.transform(X_test_features.reshape(X_test_features.shape[0], -1))

# Reshape for CNN input (batch_size, time_steps, features)
X_train_pca = X_train_pca.reshape(X_train_pca.shape[0], X_train_pca.shape[1], 1)
X_test_pca = X_test_pca.reshape(X_test_pca.shape[0], X_test_pca.shape[1], 1)

print(f"Train shape: {X_train_pca.shape}, Test shape: {X_test_pca.shape}")

Train shape: (6985, 40, 1), Test shape: (1747, 40, 1)


In [4]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import AdamW

def build_cnn_model(input_shape=(40, 1), num_classes=10):
    model = Sequential()

    # 1st CNN Layer
    model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=input_shape))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.3))

    # 2nd CNN Layer
    model.add(Conv1D(filters=128, kernel_size=3, activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.3))

    # 3rd CNN Layer
    model.add(Conv1D(filters=256, kernel_size=3, activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.3))

    # Fully connected layers
    model.add(Flatten())
    model.add(Dense(64, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.3))

    # Output layer
    model.add(Dense(num_classes, activation='softmax'))

    # Compile the model using AdamW optimizer
    optimizer = AdamW(learning_rate=0.001, weight_decay=1e-4)
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

    return model


In [5]:
from tensorflow.keras.layers import LSTM, Bidirectional

# Apply PCA for feature reduction
n_components = 40 


def build_cnn_lstm_model():
    model = Sequential()

    # 1D CNN layers with Batch Normalization
    model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(n_components, 1)))
    model.add(BatchNormalization())  # Added
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.3))

    model.add(Conv1D(filters=128, kernel_size=3, activation='relu'))
    model.add(BatchNormalization())  # Added
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.3))

    model.add(Conv1D(filters=256, kernel_size=3, activation='relu'))
    model.add(BatchNormalization())  # Added
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.3))

    # Bi-LSTM layers
    model.add(Bidirectional(LSTM(128, return_sequences=True)))
    model.add(Bidirectional(LSTM(64)))
    model.add(Dropout(0.3))

    # Fully connected layers with Batch Normalization
    model.add(Dense(64, activation='relu'))
    model.add(BatchNormalization())  # Added
    model.add(Dropout(0.3))
    model.add(Dense(10, activation='softmax'))  # 10 classes for UrbanSound8K

    # Compile the model with AdamW optimizer and weight decay
    optimizer = AdamW(learning_rate=0.001, weight_decay=1e-4)
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

    # Print model summary
    model.summary()
    return model


In [6]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.layers import MultiHeadAttention, LayerNormalization, Reshape

def build_cnn_transformer_model():
    inputs = Input(shape=(40, 1))  # Explicitly define input layer

    # CNN Feature Extraction
    x = Conv1D(filters=64, kernel_size=3, activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)

    x = Conv1D(filters=128, kernel_size=3, activation='relu')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)

    x = Conv1D(filters=256, kernel_size=3, activation='relu')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)

    # Reshape for Transformer
    x = Reshape((x.shape[1], x.shape[2]))(x)  # Ensure correct shape

    # Transformer Layer (Self-Attention)
    attn_output = MultiHeadAttention(num_heads=4, key_dim=64)(x, x)  
    x = LayerNormalization()(attn_output + x)  

    # Flatten & Fully Connected Layers
    x = Flatten()(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    outputs = Dense(10, activation='softmax')(x)  

    # Define Model
    model = Model(inputs, outputs)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

    return model

# Example usage
cnn_transformer_model = build_cnn_transformer_model()
cnn_transformer_model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 40, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 38, 64)    │        256 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 38, 64)    │        256 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 19, 64)    │          0 │ batch_normalizat… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 17, 128)   │     24,704 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 17, 128)   │        512 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 8, 128)    │          0 │ batch_normalizat… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 6, 256)    │     98,560 │ max_pooling1d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 6, 256)    │      1,024 │ conv1d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_2     │ (None, 3, 256)    │          0 │ batch_normalizat… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 3, 256)    │          0 │ max_pooling1d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 3, 256)    │    263,168 │ reshape[0][0],    │
│ (MultiHeadAttentio… │                   │            │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 3, 256)    │          0 │ multi_head_atten… │
│                     │                   │            │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 3, 256)    │        512 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 768)       │          0 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │     49,216 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 10)        │        650 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 438,858 (1.67 MB)

 Trainable params: 437,962 (1.67 MB)

 Non-trainable params: 896 (3.50 KB)

In [7]:
import numpy as np
import tensorflow.keras.backend as K
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam

# Ensure X_train_pca is a NumPy array
X_train_pca = np.array(X_train_pca)

# Ensure y_train is properly formatted
y_train = np.array(y_train)  # Convert to NumPy array
if y_train.ndim > 2:  # If too many dimensions
    y_train = y_train.squeeze()  # Remove unnecessary dimensions
num_classes = 10
y_train = to_categorical(y_train, num_classes=num_classes)

# Function to train models with dynamic learning rate adjustment
def train_model(model, X_train, y_train, initial_lr=0.001, max_epochs=50, batch_size=32, patience=5):
    """
    Trains a model while dynamically adjusting the learning rate instead of early stopping.
    """
    best_val_loss = float("inf")
    lr_factor = 0.5  # Reduction factor
    min_lr = 1e-5  # Minimum allowed learning rate

    # Set initial optimizer
    optimizer = Adam(learning_rate=initial_lr)
    model.compile(optimizer=optimizer, loss="categorical_crossentropy", metrics=["accuracy"])

    patience_counter = 0  # Track epochs without improvement

    for epoch in range(max_epochs):
        current_lr = model.optimizer.learning_rate.numpy()  # Get current LR
        print(f"\nEpoch {epoch + 1}/{max_epochs} - Learning Rate: {current_lr:.6f}")

        # Train model for one epoch
        history = model.fit(
            X_train, y_train, epochs=1, batch_size=batch_size, validation_split=0.2, verbose=1
        )

        # Get validation loss
        val_loss = history.history["val_loss"][0]

        # Check for early stopping conditions
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0  # Reset patience
        else:
            patience_counter += 1

        # If patience exceeded, reduce LR and continue
        if patience_counter >= patience:
            if current_lr > min_lr:
                new_lr = max(current_lr * lr_factor, min_lr)
                print(f"\nReducing Learning Rate to {new_lr:.6f}")
                model.optimizer.learning_rate.assign(new_lr)  # ✅ Fixed here
                patience_counter = 0  # Reset patience counter
            else:
                print("\nMinimum learning rate reached. Continuing training.")
                
    print("\nTraining Complete!\n")

# Initialize models
cnn_model = build_cnn_model()
cnn_lstm_model = build_cnn_lstm_model()
cnn_transformer_model = build_cnn_transformer_model()

# Train each model with dynamic learning rate adjustment
train_model(cnn_model, X_train_pca, y_train)
train_model(cnn_lstm_model, X_train_pca, y_train)
train_model(cnn_transformer_model, X_train_pca, y_train)


175/175 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.4651 - loss: 1.5866 - val_accuracy: 0.6034 - val_loss: 1.1768

Epoch 3/50 - Learning Rate: 0.001000
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.5106 - loss: 1.4425 - val_accuracy: 0.6356 - val_loss: 1.0991

Epoch 4/50 - Learning Rate: 0.001000
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.5650 - loss: 1.2825 - val_accuracy: 0.6557 - val_loss: 1.0270

Epoch 5/50 - Learning Rate: 0.001000
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.5762 - loss: 1.2215 - val_accuracy: 0.6743 - val_loss: 0.9900

Epoch 6/50 - Learning Rate: 0.001000
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.6205 - loss: 1.1412 - val_accuracy: 0.6750 - val_loss: 0.9723

Epoch 7/50 - Learning Rate: 0.001000
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.6345 - loss: 1.0879 - val_accuracy: 0.7115 - val_loss: 0.8907

Epoch 8/50 - Learning Rate: 0.001000
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.6453 - l

In [8]:
import numpy as np
from sklearn.metrics import classification_report, accuracy_score

# Get predictions from each model
cnn_preds = cnn_model.predict(X_test_pca)
cnn_lstm_preds = cnn_lstm_model.predict(X_test_pca)
cnn_transformer_preds = cnn_transformer_model.predict(X_test_pca)

# Average predictions for ensemble
ensemble_preds = (cnn_preds + cnn_lstm_preds + cnn_transformer_preds) / 3

# Convert predictions to class labels
y_pred_classes = np.argmax(ensemble_preds, axis=1)

# Ensure y_test is a NumPy array
y_test = np.array(y_test)

# Convert y_test to class labels if needed
y_true = y_test if y_test.ndim == 1 else np.argmax(y_test, axis=1)

# Evaluate model
print("Classification Report:")
print(classification_report(y_true, y_pred_classes))

# Print final test accuracy
ensemble_accuracy = accuracy_score(y_true, y_pred_classes)
print(f"Final Ensemble Accuracy: {ensemble_accuracy:.4f}")


55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
55/55 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step
Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.91      0.90       203
           1       0.82      0.80      0.81        86
           2       0.73      0.84      0.78       183
           3       0.79      0.75      0.77       201
           4       0.86      0.77      0.82       206
           5       0.89      0.90      0.89       193
           6       0.88      0.88      0.88        72
           7       0.86      0.90      0.88       208
           8       0.90      0.96      0.93       165
           9       0.82      0.75      0.78       230

    accuracy                           0.84      1747
   macro avg       0.84      0.84      0.84      1747
weighted avg       0.84      0.84      0.84      1747

Final Ensemble Accuracy: 0.8414
